# Lesson 2 — Prompt Templates

## Goal

understanding:
- What a prompt template is.
- Why prompt template exist.
- Why using f-string is often insufficient.
- `chatPromptTemplate`.
- `MessagesPlaceholder`.
- How prompt templates fit into the LangChain ecosystem.


In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
# Create LangChain prompt template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a senior Python instructor."),
        ("human", "Explain {topic}."), # topic is LangChain's template placeholder
    ]
)

In [3]:
print(prompt)

input_variables=['topic'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a senior Python instructor.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Explain {topic}.'), additional_kwargs={})]


In [4]:
# Render the template
prompt_value = prompt.invoke ( 
    {
        "topic": "decorators"
    }
) # Take the blueprint (prompt template) and fill in the variables

print(prompt_value)

messages=[SystemMessage(content='You are a senior Python instructor.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain decorators.', additional_kwargs={}, response_metadata={})]


In [5]:
print(type(prompt_value))

<class 'langchain_core.prompt_values.ChatPromptValue'>


In [6]:
print(prompt_value.messages)

[SystemMessage(content='You are a senior Python instructor.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain decorators.', additional_kwargs={}, response_metadata={})]


In [7]:
# Load the environment
load_dotenv()

True

In [8]:
# Read the model name
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [9]:
# Create the LangChain model
llm = ChatGoogleGenerativeAI(model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [10]:
#invoke the model
response = llm.invoke(prompt_value) # not response = llm.invoke (prompt_value.message) or response = llm.invoke (str(prompt_value)) 
# llm.invoke accepts: str or Message or List[Message] or ChatPromptValue

In [11]:
prompt = ChatPromptTemplate.from_messages( #owned by LangChain
    [
        ("system", "You are a senior Python instructor."),
        ("human", "Explain {topic} for a {level}")
    ]
)

In [12]:
prompt_value = prompt.invoke(
    { # the dictionary is owned by python
        "topic": "decorators",
        "level":"beginner"
    }
)

print(prompt_value.messages)

[SystemMessage(content='You are a senior Python instructor.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain decorators for a beginner', additional_kwargs={}, response_metadata={})]


In [13]:
prompt_value

ChatPromptValue(messages=[SystemMessage(content='You are a senior Python instructor.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain decorators for a beginner', additional_kwargs={}, response_metadata={})])

In [14]:
"""
prompt_value2 = prompt.invoke(
    { # the dictionary is owned by python
        "topic": "decorators", # will fail as prompt expect 2 keys in the dict 
    }
)
"""


'\nprompt_value2 = prompt.invoke(\n    { # the dictionary is owned by python\n        "topic": "decorators", # will fail as prompt expect 2 keys in the dict \n    }\n)\n'

In [15]:
prompt_value2 = prompt.invoke(
    { 
        "topic": "decorators",
        "level": "beginner",
        "language": "English", # the LangChain will ignores language but it will run with no problems
    }
)
prompt_value2

ChatPromptValue(messages=[SystemMessage(content='You are a senior Python instructor.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain decorators for a beginner', additional_kwargs={}, response_metadata={})])

In [16]:
print(prompt.input_variables)

['level', 'topic']


In [17]:
from langchain_core.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)

In [18]:
from langchain_core.messages import (HumanMessage, AIMessage)

history = [
    HumanMessage(
        content="What is LangChain"
    ),
    AIMessage(
        content="LangChain is a framework for building LLM applications."
    )
]

In [19]:
prompt_value = prompt.invoke(
    {
        "history": history,
        "question": "can you explain it more simply?",
    }
)

In [20]:
print(prompt_value.messages)

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is LangChain', additional_kwargs={}, response_metadata={}), AIMessage(content='LangChain is a framework for building LLM applications.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='can you explain it more simply?', additional_kwargs={}, response_metadata={})]


In [21]:
response = llm.invoke(prompt_value)
print(response)

content=[{'type': 'text', 'text': 'Imagine an AI (like ChatGPT) is a **super-smart brain**, but it is locked in a room with no internet, no memory of your past conversations, and no access to your personal files. \n\n**LangChain is like giving that brain hands, tools, and a notebook.**\n\nIt is a software tool that helps developers connect AI models to the real world. \n\nHere is what LangChain helps the AI do:\n\n1. **Connect to the Internet:** It lets the AI search Google for up-to-date information.\n2. **Read Your Files:** It lets the AI read your private PDFs, databases, or emails to answer questions about them.\n3. **Remember Things:** It gives the AI a "memory" so it remembers what you said earlier in the conversation.\n4. **Chain Tasks Together:** It allows the AI to do multi-step tasks. For example: *"First, search the web for today\'s news. Second, translate it to Spanish. Third, email it to my boss."*\n\n**In short:** If the AI is the engine, LangChain is the rest of the car 